# Station ID Mapping Check

Checks how well station IDs referenced in `raw_data/raw_trips` (`start_station_id`, `end_station_id`) map onto the reference station table in `raw_data/stations.parquet`.

**Important data-format note (discovered while building this notebook):** `stations.parquet` has *two* ID-like columns:
- `station_id`: a UUID-style string (e.g. `0825be94-1f3f-11e7-bf6b-3863bb334450`) — this is the GBFS station feed's canonical ID.
- `short_name`: a short numeric code as string (e.g. `"32324"`) — this matches the numeric IDs used in `raw_trips`.

`raw_trips.start_station_id` / `end_station_id` are numeric (e.g. `31208.0`) and correspond to `stations.short_name`, **not** `stations.station_id` (a direct check found 0 overlap between trip IDs and `station_id` UUIDs, vs. 817/855 overlap with `short_name`). This notebook therefore joins trip IDs against `stations.short_name`, while still surfacing `station_id` alongside results for reference.

Goals:
- Quantify exact-match coverage between trip station IDs and reference station IDs (`short_name`).
- Surface IDs that appear in trips but are missing from the station reference table (and vice versa).
- Report the row-level match rate (what share of trip rows have both endpoints resolvable to a known station ID).
- Compare ID-based coverage against the name-based coverage from [01_station_mapping_check.ipynb](01_station_mapping_check.ipynb) to see whether joining on ID resolves cases that name-matching missed.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
STATIONS_PATH = RAW_DATA / "stations.parquet"
RAW_TRIPS_PATH = RAW_DATA / "raw_trips"

print(f"Project root: {ROOT}")
print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")
print(f"Raw trips path: {RAW_TRIPS_PATH} (exists={RAW_TRIPS_PATH.exists()})")

## Load data

- `stations`: reference station table (`raw_data/stations.parquet`), columns include `station_id` (UUID), `short_name` (numeric/alphanumeric code — the actual trip-ID join key), `name`, `lat`, `lon`, `capacity`.
- `trips`: trip data (`raw_data/raw_trips/`), read as a full partitioned parquet dataset so any additional `year=/month=` partitions synced later are picked up automatically. Columns include `start_station_id`, `end_station_id`.

**Schema note:** `start_station_id` / `end_station_id` are stored as `int64` in older partitions (2010 through ~2020-05) but switch to a string type from 2021-02 onward, because some later station IDs are alphanumeric (e.g. `MTL-ECO5-03`, an outlier likely representing a special/placeholder station). Reading the raw partitioned dataset with pandas directly fails with `ArrowInvalid: Failed to parse string ... as a scalar of type int64` because pyarrow tries to unify all partitions under one schema inferred from the first fragment. We work around this by opening the dataset via `pyarrow.dataset` with an explicit schema that forces both ID columns to `string`, so every partition is cast consistently on read.

In [ ]:
import pyarrow as pa
import pyarrow.dataset as ds

stations = pd.read_parquet(STATIONS_PATH)

# Build the trips dataset with start/end_station_id forced to string, since their
# on-disk type changes across partitions (int64 pre-2021, alphanumeric string from 2021-02 on).
_raw_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive")
_id_string_fields = []
for field in _raw_dataset.schema:
    if field.name in ("start_station_id", "end_station_id"):
        _id_string_fields.append(pa.field(field.name, pa.string()))
    else:
        _id_string_fields.append(field)
_trips_schema = pa.schema(_id_string_fields)

trips_dataset = ds.dataset(RAW_TRIPS_PATH, format="parquet", partitioning="hive", schema=_trips_schema)
trips = trips_dataset.to_table(
    columns=[
        "start_station_id",
        "end_station_id",
        "start_station_name",
        "end_station_name",
        "period",
    ]
).to_pandas()

print(f"stations: {stations.shape[0]:,} rows, columns: {list(stations.columns)}")
print(f"trips:    {trips.shape[0]:,} rows, columns: {list(trips.columns)}")
print(f"Periods covered in trips: {sorted(trips['period'].dropna().unique())}")

print("\nstations dtypes:")
print(stations.dtypes)
print("\ntrips dtypes:")
print(trips.dtypes)

stations.head()

## Verify which stations column is the real join key

`trips.start_station_id` / `end_station_id` are now loaded as clean strings. `stations.station_id` is a UUID string and `stations.short_name` is a numeric/alphanumeric code string. Check overlap against both stations columns to confirm which one is the actual join key before proceeding.

In [ ]:
trip_start_ids_str = trips["start_station_id"].dropna()
trip_end_ids_str = trips["end_station_id"].dropna()
trip_ids = set(trip_start_ids_str.unique()) | set(trip_end_ids_str.unique())

station_id_uuids = set(stations["station_id"].dropna().unique())
station_short_names = set(stations["short_name"].dropna().unique())

overlap_with_station_id = trip_ids & station_id_uuids
overlap_with_short_name = trip_ids & station_short_names

print(f"Unique trip station IDs (start | end):                {len(trip_ids):,}")
print(f"Overlap with stations.station_id (UUIDs):             {len(overlap_with_station_id):,}")
print(f"Overlap with stations.short_name (numeric codes):     {len(overlap_with_short_name):,}")
print(
    "\n=> Using 'stations.short_name' as the join key for trip station IDs "
    "(station_id is a UUID from a different ID space and does not correspond to trip IDs)."
)

## Build the ID universes

Define the comparison sets: reference station IDs (`stations.short_name`) vs. trip station IDs (`start_station_id` ∪ `end_station_id`). Keep `station_names` (from `stations.name`) around too, for the ID/name cross-check later.

In [ ]:
station_ids = station_short_names  # reference set, already computed above
station_names = set(stations["name"].dropna().unique())  # for the ID/name cross-check later
start_ids = set(trip_start_ids_str.unique())
end_ids = set(trip_end_ids_str.unique())
# trip_ids already computed above (start_ids | end_ids)

print(f"Unique reference station IDs (stations.short_name): {len(station_ids):,}")
print(f"Unique start_station_id values:                      {len(start_ids):,}")
print(f"Unique end_station_id values:                        {len(end_ids):,}")
print(f"Unique trip station IDs (start | end):                {len(trip_ids):,}")

## Exact-match coverage

How many trip station IDs have an exact match in the station reference table (`short_name`), and how many don't? Unlike names, IDs are exact identifiers, so there's no formatting/casing ambiguity to normalize — an ID either matches or it doesn't.

In [ ]:
matched_ids = trip_ids & station_ids
trip_only_ids = trip_ids - station_ids  # in trips, missing from stations
station_only_ids = station_ids - trip_ids  # in stations, never used in trips

coverage_rate = len(matched_ids) / len(trip_ids) if trip_ids else float("nan")

print(f"Trip IDs matched exactly to a station:      {len(matched_ids):,}")
print(f"Trip IDs with NO match in stations:         {len(trip_only_ids):,}")
print(f"Station IDs never referenced in trips:      {len(station_only_ids):,}")
print(f"Exact-match coverage rate (trip IDs):       {coverage_rate:.2%}")

### Trip station IDs missing from the station reference table

These are `start_station_id` / `end_station_id` values that don't exactly match any `stations.short_name` value — potential data quality issues, retired/renumbered stations, or IDs from a period before the current live station feed.

In [ ]:
trip_only_df = pd.DataFrame(sorted(trip_only_ids, key=lambda x: (len(x), x)), columns=["trip_station_id"])
print(f"{len(trip_only_df):,} trip station IDs have no exact match in stations.parquet (short_name)")
trip_only_df.head(50)

### Reference stations never referenced in trips

Shown with `station_id` (UUID) and `name` for readability. These may be legitimate (e.g. stations added after the trip periods currently loaded) rather than data quality issues.

In [ ]:
station_only_df = (
    stations.loc[stations["short_name"].isin(station_only_ids), ["short_name", "station_id", "name"]]
    .sort_values("short_name")
    .reset_index(drop=True)
)
print(f"{len(station_only_df):,} reference stations are never referenced in the loaded trip data")
station_only_df.head(50)

## Cross-check: do unmatched IDs correspond to unmatched names too?

IDs are opaque identifiers, so "closest match" string similarity (used for names in notebook 01) doesn't apply here. Instead, check whether trip rows with an unmatched `start_station_id`/`end_station_id` also have an unmatched `start_station_name`/`end_station_name` — if the ID is bad but the name matches fine (or vice versa), that tells us which field is more reliable for joining.

In [ ]:
start_id_matched = trips["start_station_id"].isin(station_ids)
end_id_matched = trips["end_station_id"].isin(station_ids)
start_name_matched = trips["start_station_name"].isin(station_names)
end_name_matched = trips["end_station_name"].isin(station_names)

# Rows where the ID fails to match but the name matches fine, and vice versa.
start_id_bad_name_ok = (~start_id_matched) & start_name_matched
start_name_bad_id_ok = (~start_name_matched) & start_id_matched
end_id_bad_name_ok = (~end_id_matched) & end_name_matched
end_name_bad_id_ok = (~end_name_matched) & end_id_matched

print("Rows where START id is unmatched but START name IS matched: "
      f"{start_id_bad_name_ok.sum():,} ({start_id_bad_name_ok.mean():.2%})")
print("Rows where START name is unmatched but START id IS matched: "
      f"{start_name_bad_id_ok.sum():,} ({start_name_bad_id_ok.mean():.2%})")
print("Rows where END id is unmatched but END name IS matched:     "
      f"{end_id_bad_name_ok.sum():,} ({end_id_bad_name_ok.mean():.2%})")
print("Rows where END name is unmatched but END id IS matched:     "
      f"{end_name_bad_id_ok.sum():,} ({end_name_bad_id_ok.mean():.2%})")

both_id_matched = start_id_matched & end_id_matched
both_name_matched = start_name_matched & end_name_matched
print(f"\nRow-level match rate by ID (both endpoints):   {both_id_matched.mean():.2%}")
print(f"Row-level match rate by NAME (both endpoints): {both_name_matched.mean():.2%}")

## Row-level match rate (by ID)

Unique-ID coverage doesn't tell us how much of the actual trip *volume* is affected. Compute, per row, whether `start_station_id` and `end_station_id` each resolve to a known station ID, and report the share of rows fully matched on both ends.

In [ ]:
total_rows = len(trips)

start_id_match_rate = start_id_matched.mean()
end_id_match_rate = end_id_matched.mean()
both_id_match_rate = both_id_matched.mean()

print(f"Total trip rows:                                       {total_rows:,}")
print(f"Rows with start_station_id matched to a station:       {start_id_matched.sum():,} ({start_id_match_rate:.2%})")
print(f"Rows with end_station_id matched to a station:         {end_id_matched.sum():,} ({end_id_match_rate:.2%})")
print(f"Rows with BOTH endpoints matched (by id):              {both_id_matched.sum():,} ({both_id_match_rate:.2%})")
print(f"Rows with at least one endpoint unmatched (by id):     {(~both_id_matched).sum():,} ({(~both_id_matched).mean():.2%})")

### Which unmatched IDs account for the most rows?

Ranks unmatched station IDs (start or end) by how many trip rows they appear on, so we can prioritize investigating the ones with the biggest impact. Includes the corresponding station name(s) seen in trips for that ID, for context.

In [ ]:
start_id_counts = (
    trips.loc[~start_id_matched, "start_station_id"]
    .value_counts()
    .rename("as_start")
)
end_id_counts = (
    trips.loc[~end_id_matched, "end_station_id"]
    .value_counts()
    .rename("as_end")
)

unmatched_id_impact = (
    pd.concat([start_id_counts, end_id_counts], axis=1)
    .fillna(0)
    .astype(int)
)
unmatched_id_impact["total_rows"] = unmatched_id_impact["as_start"] + unmatched_id_impact["as_end"]

# Attach an example station name seen in trips for each unmatched id, for readability.
name_by_start_id = trips.loc[~start_id_matched].groupby("start_station_id")["start_station_name"].first()
name_by_end_id = trips.loc[~end_id_matched].groupby("end_station_id")["end_station_name"].first()
unmatched_id_impact["example_name"] = name_by_start_id.combine_first(name_by_end_id)

unmatched_id_impact = unmatched_id_impact.sort_values("total_rows", ascending=False)

print(f"{len(unmatched_id_impact):,} distinct unmatched IDs appear across trip rows")
print(f"They account for {unmatched_id_impact['total_rows'].sum():,} row-endpoint occurrences")
unmatched_id_impact.head(20)

## Summary

In [ ]:
from IPython.display import Markdown, display

summary = f"""
| Metric | Value |
|---|---|
| Reference station IDs (`stations.short_name`) | {len(station_ids):,} |
| Unique trip station IDs (start ∪ end) | {len(trip_ids):,} |
| Exact-match coverage (unique IDs) | {coverage_rate:.2%} |
| Trip IDs unmatched | {len(trip_only_ids):,} |
| Reference stations never used in trips (by id) | {len(station_only_ids):,} |
| Trip rows | {total_rows:,} |
| Row-level match rate by ID (both endpoints) | {both_id_match_rate:.2%} |
| Row-level match rate by NAME (both endpoints, for comparison) | {both_name_matched.mean():.2%} |

**Note:** `stations.station_id` (UUID) is a different ID space and was found to have 0 overlap with trip station IDs — `stations.short_name` is the correct join key. See [01_station_mapping_check.ipynb](01_station_mapping_check.ipynb) for the name-based version of this check.
"""

display(Markdown(summary))